# Nepal Hydropower Mapping and Spatial Analysis

Google Colab workflow for mapping hydropower stations in Nepal by installed capacity. It produces a publication-style static map, an interactive Folium map, cleaned spatial exports, and summary statistics.

**Before running:** prepare a CSV and a complete Nepal boundary shapefile (`.shp`, `.shx`, `.dbf`, `.prj`). Check the required CSV column names in the README.

In [ ]:
# ============================================
# NEPAL HYDROPOWER - PUBLICATION READY MAP
# Google Colab Notebook
# ============================================


In [ ]:
# STEP 1: Install required packages
!pip install geopandas matplotlib contextily mapclassify adjustText matplotlib-scalebar -q


In [ ]:
# STEP 2: Import libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib_scalebar.scalebar import ScaleBar
import numpy as np
from shapely.geometry import Point
import contextily as cx
from adjustText import adjust_text
import warnings
warnings.filterwarnings('ignore')

# Set publication-quality defaults
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5

print("✅ Libraries installed and configured")

# ============================================


In [ ]:
# STEP 3: UPLOAD YOUR FILES
# ============================================
# Run this cell, then upload your files when prompted
from google.colab import files

print("📁 Upload your CSV file with hydropower data...")
uploaded_csv = files.upload()
csv_filename = list(uploaded_csv.keys())[0]

print("\n📁 Upload your Nepal boundary Shapefile (and associated files)...")
print("Note: Upload all files: .shp, .shx, .dbf, .prj")
uploaded_shp = files.upload()
shp_filename = [k for k in uploaded_shp.keys() if k.endswith('.shp')][0]

print(f"\n✅ Uploaded: {csv_filename} and {shp_filename}")

# ============================================


In [ ]:
# STEP 4: LOAD AND PROCESS DATA
# ============================================

# Load CSV data
df = pd.read_csv(csv_filename)
print("CSV Columns:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

# Clean column names (remove spaces and special characters)
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('Â', '')

# Ensure numeric columns
df['Capacity_(MW)'] = pd.to_numeric(df['Capacity_(MW)'], errors='coerce')
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Remove rows with missing coordinates or capacity
df_clean = df.dropna(subset=['Latitude', 'Longitude', 'Capacity_(MW)'])

# Create geometry
geometry = [Point(xy) for xy in zip(df_clean['Longitude'], df_clean['Latitude'])]
gdf_hydro = gpd.GeoDataFrame(
    df_clean,
    geometry=geometry,
    crs='EPSG:4326'  # WGS84
)

# Load Nepal boundary
gdf_nepal = gpd.read_file(shp_filename)
if gdf_nepal.crs is None:
    gdf_nepal.set_crs('EPSG:4326', inplace=True)
else:
    gdf_nepal = gdf_nepal.to_crs('EPSG:4326')

print(f"\n✅ Loaded {len(gdf_hydro)} hydropower stations")
print(f"✅ Nepal boundary loaded: {len(gdf_nepal)} features")

# ============================================


In [ ]:
# STEP 5: CLASSIFICATION FOR VISUALIZATION
# ============================================

def classify_capacity(cap):
    """Standard hydropower classification"""
    if cap < 10:
        return 'Micro (<10 MW)'
    elif cap < 25:
        return 'Small (10-25 MW)'
    elif cap < 100:
        return 'Medium (25-100 MW)'
    else:
        return 'Large (>100 MW)'

def get_symbol_size(cap):
    """
    Graduated symbol sizes using perceptual scaling
    Based on cartographic principles: area ∝ value
    """
    if cap < 10:
        return 50      # Micro
    elif cap < 25:
        return 150     # Small
    elif cap < 100:
        return 400     # Medium
    else:
        return 900     # Large

# Apply classifications
gdf_hydro['Capacity_Class'] = gdf_hydro['Capacity_(MW)'].apply(classify_capacity)
gdf_hydro['Symbol_Size'] = gdf_hydro['Capacity_(MW)'].apply(get_symbol_size)

# Color scheme - ColorBrewer 2.0 Set2 (qualitative, colorblind-safe)
# Using different colors for different provinces or capacity classes
colors = {
    'Micro (<10 MW)': '#66c2a5',      # Teal
    'Small (10-25 MW)': '#fc8d62',    # Orange
    'Medium (25-100 MW)': '#8da0cb',   # Blue
    'Large (>100 MW)': '#e78ac3'       # Pink
}

gdf_hydro['Color'] = gdf_hydro['Capacity_Class'].map(colors)

print("\n📊 Capacity Distribution:")
print(gdf_hydro['Capacity_Class'].value_counts())

# ============================================


In [ ]:
# STEP 6: CREATE PUBLICATION-READY MAP
# ============================================

# Reproject to UTM Zone 45N for accurate scale (EPSG:32645)
gdf_hydro_utm = gdf_hydro.to_crs('EPSG:32645')
gdf_nepal_utm = gdf_nepal.to_crs('EPSG:32645')

# Create figure with journal-standard dimensions
# A4 width = 210mm, 2-column figure = 89mm (~3.5 inches)
fig, ax = plt.subplots(figsize=(7, 9), dpi=300)  # Portrait for Nepal's shape

# Plot Nepal boundary (subtle)
gdf_nepal_utm.boundary.plot(
    ax=ax,
    color='#999999',
    linewidth=0.8,
    alpha=0.6,
    zorder=1
)

# Plot hydropower stations with graduated symbols
# Sort by size so larger symbols are plotted first (underneath)
gdf_sorted = gdf_hydro_utm.sort_values('Symbol_Size', ascending=False)

scatter = ax.scatter(
    gdf_sorted.geometry.x,
    gdf_sorted.geometry.y,
    s=gdf_sorted['Symbol_Size'],
    c=gdf_sorted['Color'],
    alpha=0.85,
    edgecolors='#333333',
    linewidths=0.5,
    zorder=3
)

# Add labels for large stations (>100 MW) only
large_stations = gdf_hydro_utm[gdf_hydro_utm['Capacity_(MW)'] > 100]
texts = []
for idx, row in large_stations.iterrows():
    txt = ax.annotate(
        row['Name_of_station'],
        (row.geometry.x, row.geometry.y),
        fontsize=6,
        fontweight='bold',
        ha='center',
        va='bottom',
        xytext=(0, 10),
        textcoords='offset points',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'),
        zorder=4
    )
    texts.append(txt)

# Adjust text to avoid overlap
adjust_text(texts, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

# Add scale bar
scalebar = ScaleBar(
    1,  # 1 meter
    location='lower right',
    box_alpha=0.8,
    scale_loc='bottom',
    length_fraction=0.25,
    width_fraction=0.02,
    font_properties={'size': 8}
)
ax.add_artist(scalebar)

# Add north arrow
x, y, arrow_length = 0.95, 0.95, 0.08
ax.annotate(
    'N',
    xy=(x, y),
    xytext=(x, y - arrow_length),
    arrowprops=dict(facecolor='black', width=3, headwidth=8),
    ha='center',
    va='center',
    fontsize=12,
    fontweight='bold',
    xycoords=ax.transAxes
)

# Create custom legend
legend_elements = [
    mpatches.Patch(facecolor=colors['Micro (<10 MW)'], edgecolor='#333333',
                   label='Micro (<10 MW) • Small circle'),
    mpatches.Patch(facecolor=colors['Small (10-25 MW)'], edgecolor='#333333',
                   label='Small (10-25 MW) • Medium circle'),
    mpatches.Patch(facecolor=colors['Medium (25-100 MW)'], edgecolor='#333333',
                   label='Medium (25-100 MW) • Large circle'),
    mpatches.Patch(facecolor=colors['Large (>100 MW)'], edgecolor='#333333',
                   label='Large (>100 MW) • Extra large circle')
]

legend = ax.legend(
    handles=legend_elements,
    loc='upper left',
    title='Hydropower Capacity',
    title_fontsize=9,
    fontsize=7,
    frameon=True,
    fancybox=False,
    edgecolor='#cccccc',
    facecolor='white',
    framealpha=0.9
)

# Clean up axes
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')

# Remove spines
for spine in ax.spines.values():
    spine.set_visible(False)

# Add subtle frame
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['left'].set_visible(True)
for spine in ax.spines.values():
    spine.set_linewidth(0.5)
    spine.set_color('#cccccc')

# Title
ax.set_title(
    'Nepal Hydropower Stations by Capacity Class\n',
    fontsize=11,
    fontweight='bold',
    pad=10
)

# Add statistics text
stats_text = (f"Total Stations: {len(gdf_hydro)}\n"
              f"Total Capacity: {gdf_hydro['Capacity_(MW)'].sum():.0f} MW\n"
              f"Largest: {gdf_hydro.loc[gdf_hydro['Capacity_(MW)'].idxmax(), 'Name_of_station']} "
              f"({gdf_hydro['Capacity_(MW)'].max():.0f} MW)")

ax.text(
    0.02, 0.02,
    stats_text,
    transform=ax.transAxes,
    fontsize=7,
    verticalalignment='bottom',
    horizontalalignment='left',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none')
)

plt.tight_layout()

# Save high-resolution output
output_path = '/content/nepal_hydropower_publication.png'
plt.savefig(
    output_path,
    dpi=600,
    bbox_inches='tight',
    pad_inches=0.1,
    facecolor='white',
    edgecolor='none'
)

print(f"\n✅ Map saved to: {output_path}")
plt.show()

# Download the file
files.download(output_path)

# ============================================


In [ ]:
# STEP 7: ALTERNATIVE - INTERACTIVE HTML MAP
# ============================================

# Create interactive map with Folium (optional)
import folium
from folium.plugins import MarkerCluster

# Create base map centered on Nepal
m = folium.Map(
    location=[28.2, 84.5],
    zoom_start=7,
    tiles='CartoDB positron',  # Clean, light base map
    control_scale=True
)

# Add Nepal boundary
folium.GeoJson(
    gdf_nepal,
    style_function=lambda x: {
        'fillColor': 'none',
        'color': '#999999',
        'weight': 1,
        'fillOpacity': 0
    },
    name='Nepal Boundary'
).add_to(m)

# Add circle markers for each station
for idx, row in gdf_hydro.iterrows():
    # Determine size and color
    size = get_symbol_size(row['Capacity_(MW)']) / 10  # Scale for web
    color = colors[row['Capacity_Class']]

    # Create popup content
    popup_html = f"""
    <b>{row['Name_of_station']}</b><br>
    Province: {row['Province']}<br>
    Capacity: {row['Capacity_(MW)']} MW<br>
    Class: {row['Capacity_Class']}<br>
    Commissioned: {row.get('Commissioned', 'N/A')}
    """

    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=np.sqrt(size),
        popup=folium.Popup(popup_html, max_width=200),
        color='#333333',
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        tooltip=row['Name_of_station']
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed;
            bottom: 50px; left: 50px; width: 180px;
            background-color: white; border: 1px solid grey;
            border-radius: 5px; padding: 10px; font-size: 12px;">
    <b>Capacity Class</b><br>
    <i style="background:#66c2a5; width:12px; height:12px; display:inline-block;"></i> Micro (<10 MW)<br>
    <i style="background:#fc8d62; width:12px; height:12px; display:inline-block;"></i> Small (10-25 MW)<br>
    <i style="background:#8da0cb; width:12px; height:12px; display:inline-block;"></i> Medium (25-100 MW)<br>
    <i style="background:#e78ac3; width:12px; height:12px; display:inline-block;"></i> Large (>100 MW)
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Save interactive map
html_path = '/content/nepal_hydropower_interactive.html'
m.save(html_path)
print(f"\n✅ Interactive map saved to: {html_path}")
files.download(html_path)

# ============================================


In [ ]:
# STEP 8: EXPORT SHAPEFILE FOR ARCMAP
# ============================================

# Export cleaned data as shapefile for ArcMap
export_gdf = gdf_hydro.copy()
export_gdf = export_gdf[[
    'Name_of_station', 'Province', 'Capacity_(MW)',
    'Capacity_Class', 'Commissioned', 'Owner',
    'Latitude', 'Longitude', 'geometry'
]]

# Save to GeoPackage (better than shapefile, preserves field names)
gpkg_path = '/content/nepal_hydropower_cleaned.gpkg'
export_gdf.to_file(gpkg_path, driver='GPKG')

# Also save as CSV with coordinates
csv_export = export_gdf.copy()
csv_export['Lon'] = csv_export.geometry.x
csv_export['Lat'] = csv_export.geometry.y
csv_export.drop('geometry', axis=1).to_csv('/content/nepal_hydropower_cleaned.csv', index=False)

print(f"\n✅ Cleaned data exported:")
print(f"   - GeoPackage: {gpkg_path}")
print(f"   - CSV: /content/nepal_hydropower_cleaned.csv")

# Download files
files.download(gpkg_path)
files.download('/content/nepal_hydropower_cleaned.csv')

# ============================================


In [ ]:
# STEP 9: SUMMARY STATISTICS
# ============================================

print("\n" + "="*50)
print("PUBLICATION SUMMARY STATISTICS")
print("="*50)

print(f"\nTotal Hydropower Stations: {len(gdf_hydro)}")
print(f"Total Installed Capacity: {gdf_hydro['Capacity_(MW)'].sum():.1f} MW")

print("\nCapacity by Class:")
class_stats = gdf_hydro.groupby('Capacity_Class').agg({
    'Capacity_(MW)': ['count', 'sum', 'mean']
}).round(1)
print(class_stats)

print("\nTop 5 Largest Stations:")
top5 = gdf_hydro.nlargest(5, 'Capacity_(MW)')[['Name_of_station', 'Province', 'Capacity_(MW)']]
print(top5.to_string(index=False))

print("\nProvince-wise Distribution:")
prov_stats = gdf_hydro.groupby('Province')['Capacity_(MW)'].agg(['count', 'sum']).round(1)
prov_stats.columns = ['Stations', 'Total_MW']
print(prov_stats.sort_values('Total_MW', ascending=False))

print("\n" + "="*50)
print("✅ ANALYSIS COMPLETE")
print("="*50)
